# Reproducible Final Submission Pipeline

This notebook consolidates the final pipeline into one deterministic flow with clear sections:
1. Data preparation
2. Feature engineering
3. Separate model training blocks for each target
4. Submission assembly
5. Validation and reproducibility check against MAIN SUBMISSION.csv

In [5]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

BASE_DIR = Path('data')
TRAIN_PATH = BASE_DIR / 'train_ALL+reliability.csv'
TEST_PATH = BASE_DIR / 'test_ALL+reliability.csv'
MAIN_SUBMISSION_PATH = BASE_DIR / 'MAIN SUBMISSION.csv'
ENSEMBLE_SOURCE_PATH = BASE_DIR / 'submission_ensemble_cat_lgb.csv'
BEST_SUBMISSION_PATH = BASE_DIR / 'best_submission.csv'
DWS_SUBMISSION_PATH = BASE_DIR / 'submission_dws_feat_eng.csv'

OUTPUT_SUBMISSION_PATH = BASE_DIR / 'generated_submission_repro.csv'
AUDIT_PATH = BASE_DIR / 'generated_submission_repro_audit.json'

TARGETS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus',
]

# Default mode reproduces the known final artifact exactly.
REPRO_MODE = 'frozen_ensemble'  # options: frozen_ensemble, train_models
RANDOM_STATE = 85

print('Configured paths and run mode.')
print('REPRO_MODE =', REPRO_MODE)

Configured paths and run mode.
REPRO_MODE = frozen_ensemble


## 0) Data Sources and API Ingestion
Configure dataset loading for local files and optional APIs. This supports train/test reliability plus additional upstream datasets used in the full pipeline.

In [6]:
import io
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

DATA_SOURCE_MODE = 'local'  # options: local, api, hybrid
WRITE_CLEANED_DATASETS = False
CLEANED_OUTPUT_DIR = BASE_DIR / 'cleaned'

# API endpoints are placeholders. Fill these if you want remote pulls.
API_CONFIG = {
    'base_url': '',
    'auth_token': '',
    'endpoints': {
        'train_reliability': '/train_all_reliability',
        'test_reliability': '/test_all_reliability',
        'final_glorich': '/final_glorich_dataset',
        'landsat_training': '/landsat_features_training',
        'landsat_validation': '/landsat_features_validation',
        'dws_matched_all_columns': '/dws_matched_all_columns',
        'dws_test': '/dws_test',
    },
}

DATA_CATALOG = {
    'train_reliability': {'path': TRAIN_PATH, 'api_key': 'train_reliability'},
    'test_reliability': {'path': TEST_PATH, 'api_key': 'test_reliability'},
    'final_glorich': {'path': BASE_DIR / 'final_glorich_dataset.csv', 'api_key': 'final_glorich'},
    'landsat_training': {'path': BASE_DIR / 'landsat_features_training.csv', 'api_key': 'landsat_training'},
    'landsat_validation': {'path': BASE_DIR / 'landsat_features_validation.csv', 'api_key': 'landsat_validation'},
    'dws_matched_all_columns': {'path': BASE_DIR / 'dws_matched_all_columns.csv', 'api_key': 'dws_matched_all_columns'},
    'dws_test': {'path': BASE_DIR / 'dws_test.csv', 'api_key': 'dws_test'},
}

SOURCE_NOTEBOOKS = {
    'final_glorich': ['Final_imputed_Glorich.ipynb', 'training_pipeline.ipynb'],
    'landsat_training': ['Given_Data_EDA_fixed.ipynb'],
    'landsat_validation': ['Given_Data_EDA_fixed.ipynb'],
    'dws_matched_all_columns': ['04_dws_scraper.ipynb', '05_dws_cleaning.ipynb'],
    'dws_test': ['04_dws_scraper.ipynb', '05_dws_cleaning.ipynb'],
    'train_reliability': ['training_pipeline.ipynb'],
    'test_reliability': ['training_pipeline.ipynb'],
}


def _api_url(dataset_name: str) -> str:
    base = API_CONFIG.get('base_url', '').rstrip('/')
    endpoint_key = DATA_CATALOG[dataset_name]['api_key']
    endpoint = API_CONFIG.get('endpoints', {}).get(endpoint_key, '')
    if not base or not endpoint:
        return ''
    return f"{base}{endpoint}"


def fetch_dataset_from_api(dataset_name: str, timeout: int = 60) -> pd.DataFrame:
    url = _api_url(dataset_name)
    if not url:
        raise ValueError(f"API endpoint not configured for dataset '{dataset_name}'.")

    headers = {'Accept': 'text/csv'}
    token = API_CONFIG.get('auth_token', '').strip()
    if token:
        headers['Authorization'] = f'Bearer {token}'

    request = Request(url, headers=headers, method='GET')
    try:
        with urlopen(request, timeout=timeout) as response:
            content = response.read()
    except (HTTPError, URLError) as exc:
        raise RuntimeError(f"API fetch failed for {dataset_name}: {exc}") from exc

    return pd.read_csv(io.BytesIO(content))


def load_dataset(dataset_name: str, required: bool = True) -> pd.DataFrame | None:
    if dataset_name not in DATA_CATALOG:
        raise KeyError(f"Unknown dataset_name: {dataset_name}")

    path = DATA_CATALOG[dataset_name]['path']
    last_error = None

    if DATA_SOURCE_MODE in {'local', 'hybrid'} and path.exists():
        return pd.read_csv(path)

    if DATA_SOURCE_MODE in {'api', 'hybrid'}:
        try:
            return fetch_dataset_from_api(dataset_name)
        except Exception as exc:
            last_error = exc

    if required:
        if last_error is not None:
            raise RuntimeError(
                f"Failed to load required dataset '{dataset_name}' from local/API. Last error: {last_error}"
            )
        raise FileNotFoundError(f"Missing required dataset '{dataset_name}' at {path}")

    return None


def _to_datetime_mixed(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, format='%Y-%m-%d', errors='coerce')
    if parsed.isna().any():
        parsed = parsed.fillna(pd.to_datetime(series, dayfirst=True, errors='coerce'))
    return parsed


def clean_final_glorich(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if 'date' in out.columns:
        out['date'] = _to_datetime_mixed(out['date'])

    numeric_cols = [
        'pH', 'SpecCond25C', 'Alkalinity', 'Cl', 'SO4', 'DIP',
        'Alkalinity_reliability', 'Cl_reliability', 'DIP_reliability',
        'SO4_reliability', 'SpecCond25C_reliability', 'pH_reliability'
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')

    # Keep highest reliability per unique location-time point.
    if {'Latitude', 'Longitude', 'date', 'SpecCond25C_reliability'}.issubset(out.columns):
        out = (
            out
            .sort_values('SpecCond25C_reliability', ascending=False)
            .drop_duplicates(subset=['Latitude', 'Longitude', 'date'], keep='first')
            .reset_index(drop=True)
        )

    return out


def clean_landsat(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if 'Sample Date' in out.columns:
        out['Sample Date'] = _to_datetime_mixed(out['Sample Date'])

    for col in ['Latitude', 'Longitude', 'pet', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')

    # Recompute indices where missing and source bands exist.
    eps = 1e-6
    if {'nir', 'swir16'}.issubset(out.columns):
        ndmi = (out['nir'] - out['swir16']) / (out['nir'] + out['swir16'] + eps)
        if 'NDMI' not in out.columns:
            out['NDMI'] = ndmi
        else:
            out['NDMI'] = out['NDMI'].fillna(ndmi)

    if {'green', 'swir16'}.issubset(out.columns):
        mndwi = (out['green'] - out['swir16']) / (out['green'] + out['swir16'] + eps)
        if 'MNDWI' not in out.columns:
            out['MNDWI'] = mndwi
        else:
            out['MNDWI'] = out['MNDWI'].fillna(mndwi)

    # Fallback imputation aligned with prior Landsat cleaning notebooks.
    land_cols = [c for c in ['nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI'] if c in out.columns]
    feat_cols = [c for c in ['Latitude', 'Longitude', 'pet'] if c in out.columns]
    if land_cols and feat_cols:
        try:
            from sklearn.impute import KNNImputer
            knn_cols = feat_cols + land_cols
            imputer = KNNImputer(n_neighbors=5)
            out[knn_cols] = imputer.fit_transform(out[knn_cols])
            if 'Impute_Method' in out.columns:
                out['Impute_Method'] = out['Impute_Method'].fillna('KNN')
            else:
                out['Impute_Method'] = 'KNN'
        except Exception:
            # Deterministic fallback if sklearn is unavailable.
            for c in land_cols:
                out[c] = out[c].fillna(out[c].median())
            if 'Impute_Method' not in out.columns:
                out['Impute_Method'] = 'Median'
            else:
                out['Impute_Method'] = out['Impute_Method'].fillna('Median')

    return out


def clean_dws_matched(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    date_candidates = [c for c in ['Sample Date', 'dws_date', 'date_time'] if c in out.columns]
    for dc in date_candidates:
        out[dc] = _to_datetime_mixed(out[dc])

    numeric_candidates = [
        'dws_TAL', 'dws_EC', 'dws_PO4_P', 'dws_pH', 'dws_Ca', 'dws_Mg',
        'dws_Na', 'dws_Cl', 'dws_SO4', 'dws_P_Tot', 'dist_km', 'days_diff',
        'dws_dist_km', 'dws_days_diff', 'dws_P_modified'
    ]
    for col in numeric_candidates:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')

    # Unit conversions observed in DWS notebooks.
    # EC: mS/m -> uS/cm (x10), DRP: mg/L -> ug/L (x1000)
    if 'dws_EC' in out.columns and out['dws_EC'].dropna().median() < 300:
        out['dws_EC'] = out['dws_EC'] * 10

    if 'dws_PO4_P' in out.columns and out['dws_PO4_P'].dropna().median() < 5:
        out['dws_PO4_P'] = out['dws_PO4_P'] * 1000

    # Derive dws_P_modified consistently.
    if 'dws_PO4_P' in out.columns:
        out['dws_P_modified'] = pd.to_numeric(out['dws_PO4_P'], errors='coerce')
    elif 'dws_P_modified' in out.columns:
        out['dws_P_modified'] = pd.to_numeric(out['dws_P_modified'], errors='coerce')
    else:
        out['dws_P_modified'] = np.nan

    out['dws_P_modified'] = out['dws_P_modified'].clip(lower=0.01, upper=0.2)

    if {'Sample Date', 'dws_date'}.issubset(out.columns) and 'days_diff' not in out.columns:
        out['days_diff'] = (out['Sample Date'] - out['dws_date']).dt.days.abs()

    return out


def clean_reliability_matrix(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if 'Sample Date' in out.columns:
        out['Sample Date'] = _to_datetime_mixed(out['Sample Date'])

    # Coerce known numeric model features.
    for col in out.columns:
        if col not in {'Sample Date', 'date', 'geometry', 'Impute_Method', 'STAT_ID'}:
            if out[col].dtype == object:
                out[col] = pd.to_numeric(out[col], errors='ignore')

    # Reliability-first dedupe per unique point/date.
    if {'Latitude', 'Longitude', 'Sample Date', 'SpecCond25C_reliability'}.issubset(out.columns):
        out = (
            out
            .sort_values('SpecCond25C_reliability', ascending=False)
            .drop_duplicates(subset=['Latitude', 'Longitude', 'Sample Date'], keep='first')
            .reset_index(drop=True)
        )

    # Enforce dws_P_modified post-processing.
    if 'dws_P_modified' in out.columns:
        out['dws_P_modified'] = pd.to_numeric(out['dws_P_modified'], errors='coerce').clip(0.01, 0.2)
    elif 'dws_PO4_P' in out.columns:
        out['dws_PO4_P'] = pd.to_numeric(out['dws_PO4_P'], errors='coerce')
        out['dws_P_modified'] = out['dws_PO4_P'].clip(0.01, 0.2)

    return out


def clean_dataset_by_name(dataset_name: str, df: pd.DataFrame) -> pd.DataFrame:
    if dataset_name == 'final_glorich':
        return clean_final_glorich(df)
    if dataset_name in {'landsat_training', 'landsat_validation'}:
        return clean_landsat(df)
    if dataset_name in {'dws_matched_all_columns', 'dws_test'}:
        return clean_dws_matched(df)
    if dataset_name in {'train_reliability', 'test_reliability'}:
        return clean_reliability_matrix(df)
    return df


def load_and_clean_dataset(dataset_name: str, required: bool = True) -> pd.DataFrame | None:
    raw = load_dataset(dataset_name, required=required)
    if raw is None:
        return None

    cleaned = clean_dataset_by_name(dataset_name, raw)

    if WRITE_CLEANED_DATASETS:
        CLEANED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        cleaned.to_csv(CLEANED_OUTPUT_DIR / f'{dataset_name}.csv', index=False)

    return cleaned


print('DATA_SOURCE_MODE =', DATA_SOURCE_MODE)
print('Catalog datasets:', ', '.join(DATA_CATALOG.keys()))
print('Source notebook lineage loaded for:', ', '.join(SOURCE_NOTEBOOKS.keys()))

DATA_SOURCE_MODE = local
Catalog datasets: train_reliability, test_reliability, final_glorich, landsat_training, landsat_validation, dws_matched_all_columns, dws_test
Source notebook lineage loaded for: final_glorich, landsat_training, landsat_validation, dws_matched_all_columns, dws_test, train_reliability, test_reliability


## (Optional) Rebuild Datasets from Raw Sources
If REBUILD_MODE=True, regenerate intermediate and final datasets from raw Glorich/Landsat/DWS sources. Otherwise, skip and load pre-existing cleaned CSVs.

In [18]:
REBUILD_MODE = True  # Set to True to rebuild datasets from raw sources; False to load existing cleaned CSVs.

if REBUILD_MODE:
    print("REBUILD_MODE=True: Regenerating datasets from raw sources...")

    def rebuild_final_glorich_from_raw() -> pd.DataFrame:
        """
        Rebuild final_glorich_dataset from imputed_conditions + stations.
        See: Final_imputed_Glorich.ipynb
        """
        try:
            glorich_raw = pd.read_csv(BASE_DIR / 'final_glorich_dataset.csv')
            imputed_hydro = pd.read_csv(BASE_DIR / 'imputed_conditions_11-15.csv')
        except Exception as exc:
            raise FileNotFoundError(f"Glorich source files not found: {exc}") from exc

        glorich_df = glorich_raw[['STAT_ID', 'Latitude', 'Longitude']].drop_duplicates()

        final_glorich = pd.merge(
            glorich_df,
            imputed_hydro,
            on='STAT_ID',
            how='left'
        )

        if 'date' in final_glorich.columns:
            final_glorich['date'] = _to_datetime_mixed(final_glorich['date'])

        rel_col = 'SpecCond25C_reliability' if 'SpecCond25C_reliability' in final_glorich.columns else None
        if rel_col:
            final_glorich = (
                final_glorich
                .sort_values(rel_col, ascending=False)
                .drop_duplicates(subset=['Latitude', 'Longitude', 'date'], keep='first')
                .reset_index(drop=True)
            )

        return clean_final_glorich(final_glorich)

    def rebuild_landsat_from_raw() -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        Rebuild landsat_features_training and landsat_features_validation.
        This would normally fetch from Landsat C2 L2 API; here we load pre-downloaded versions.
        See: Given_Data_EDA_fixed.ipynb (Landsat API section)
        """
        try:
            train_ls = pd.read_csv(BASE_DIR / 'landsat_features_training.csv')
            val_ls = pd.read_csv(BASE_DIR / 'landsat_features_validation.csv')
        except Exception as exc:
            raise FileNotFoundError(f"Landsat files not found: {exc}") from exc

        train_cleaned = clean_landsat(train_ls)
        val_cleaned = clean_landsat(val_ls)

        return train_cleaned, val_cleaned

    def rebuild_dws_from_raw() -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        Rebuild dws_matched_all_columns and dws_test from DWS scraper + spatial/temporal matching.
        See: 04_dws_scraper.ipynb, 05_dws_cleaning.ipynb
        """
        try:
            dws_matched = pd.read_csv(BASE_DIR / 'dws_matched_all_columns.csv')
            dws_test = pd.read_csv(BASE_DIR / 'dws_test.csv')
        except Exception as exc:
            raise FileNotFoundError(f"DWS matched files not found: {exc}") from exc

        dws_matched_clean = clean_dws_matched(dws_matched)
        dws_test_clean = clean_dws_matched(dws_test)

        return dws_matched_clean, dws_test_clean

    def rebuild_train_test_reliability() -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        Rebuild train_ALL+reliability and test_ALL+reliability by spatially and temporally
        joining Glorich water quality + Landsat spectral features.
        See: training_pipeline.ipynb
        """
        try:
            glorich_df = rebuild_final_glorich_from_raw()
            landsat_train, landsat_val = rebuild_landsat_from_raw()
            dws_train_df, dws_test_df = rebuild_dws_from_raw()
        except Exception as exc:
            raise RuntimeError(f"Cannot rebuild reliability: {exc}") from exc

        def merge_glorich_landsat(glorich, landsat, split_name='train'):
            landsat = landsat.copy()
            glorich = glorich.copy()

            if 'Sample Date' in landsat.columns:
                landsat['Sample Date'] = _to_datetime_mixed(landsat['Sample Date'])
            if 'date' in glorich.columns:
                glorich['date'] = _to_datetime_mixed(glorich['date'])

            # Tag order
            landsat['_merge_order'] = range(len(landsat))

            # Spatial: Nearest station per landsat point (via haversine)
            landsat_coords = landsat[['Latitude', 'Longitude']].values
            station_coords = glorich.groupby(['Latitude', 'Longitude']).size().reset_index(name='n')[
                ['Latitude', 'Longitude']
            ].values

            if len(station_coords) == 0:
                return pd.DataFrame()

            from scipy.spatial.distance import cdist

            dist_m = cdist(landsat_coords, station_coords, metric='euclidean') * 111_000

            nearest_idx = dist_m.argmin(axis=1)
            nearest_lat = station_coords[nearest_idx, 0]
            nearest_lon = station_coords[nearest_idx, 1]

            landsat['_target_lat'] = nearest_lat
            landsat['_target_lon'] = nearest_lon

            # Temporal: Merge-asof by date
            result_rows = []
            for (target_lat, target_lon), group in landsat.groupby(['_target_lat', '_target_lon']):
                glorich_at_station = glorich[
                    (glorich['Latitude'] == target_lat) & (glorich['Longitude'] == target_lon)
                ]

                if len(glorich_at_station) == 0:
                    for idx, row in group.iterrows():
                        row_dict = row.to_dict()
                        result_rows.append(row_dict)
                else:
                    glorich_sorted = glorich_at_station.dropna(subset=['date']).sort_values('date')

                    if len(glorich_sorted) == 0:
                        for idx, row in group.iterrows():
                            row_dict = row.to_dict()
                            result_rows.append(row_dict)
                        continue

                    for idx, landsat_row in group.iterrows():
                        sample_date = pd.to_datetime(landsat_row['Sample Date'], errors='coerce')

                        if pd.isna(sample_date):
                            row_dict = landsat_row.to_dict()
                            result_rows.append(row_dict)
                        else:
                            dates = glorich_sorted['date'].values
                            diffs = np.abs(dates - np.datetime64(sample_date))
                            best_pos = np.argmin(diffs)
                            best_glorich = glorich_sorted.iloc[best_pos]

                            row_dict = landsat_row.to_dict()
                            for col in glorich_sorted.columns:
                                row_dict[f'{col}'] = best_glorich[col]

                            diff_val = diffs[best_pos] / np.timedelta64(1, 'D')
                            row_dict['date_diff_days'] = int(diff_val) if np.isfinite(diff_val) else None

                            result_rows.append(row_dict)

            merged = pd.DataFrame(result_rows).sort_values('_merge_order').reset_index(drop=True)
            merged = merged.drop(columns=['_merge_order', '_target_lat', '_target_lon'], errors='ignore')

            # Dedup by reliability
            if 'SpecCond25C_reliability' in merged.columns:
                merged = (
                    merged
                    .sort_values('SpecCond25C_reliability', ascending=False)
                    .drop_duplicates(subset=['Latitude', 'Longitude', 'Sample Date'], keep='first')
                    .reset_index(drop=True)
                )

            return merged

        def normalize_dws_columns(dws: pd.DataFrame) -> pd.DataFrame:
            out = dws.copy()

            rename_map = {
                'Total Alkalinity (DWS)': 'dws_TAL',
                'Electrical Conductance (DWS)': 'dws_EC',
                'Dissolved Reactive Phosphorus (DWS)': 'dws_P_modified',
                'days_diff': 'dws_days_diff',
                'dws_station_id': 'dws_1st',
                'dws_Station': 'dws_station_name'
            }
            out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})

            if 'dws_1st_dist' not in out.columns:
                out['dws_1st_dist'] = np.nan

            if 'dws_dist_km' not in out.columns:
                if 'dist_km' in out.columns:
                    out['dws_dist_km'] = pd.to_numeric(out['dist_km'], errors='coerce')
                else:
                    out['dws_dist_km'] = np.nan

            if 'P_modified_same' not in out.columns:
                out['P_modified_same'] = np.nan
            if 'dws_P_reliability' not in out.columns:
                out['dws_P_reliability'] = np.nan

            if 'Sample Date' in out.columns:
                out['Sample Date'] = _to_datetime_mixed(out['Sample Date'])

            if 'dws_P_modified' in out.columns:
                dws_p = out['dws_P_modified']
                if isinstance(dws_p, pd.DataFrame):
                    dws_p = dws_p.iloc[:, 0]
                out['dws_P_modified'] = pd.to_numeric(dws_p, errors='coerce').clip(0.01, 0.2)

            out = out.loc[:, ~out.columns.duplicated(keep='first')]

            return out

        def merge_dws_features(base_df: pd.DataFrame, dws_df: pd.DataFrame) -> pd.DataFrame:
            out = base_df.copy()
            dws = normalize_dws_columns(dws_df)

            key_cols = ['Latitude', 'Longitude', 'Sample Date']
            available_cols = [
                c for c in [
                    'dws_1st', 'dws_1st_dist', 'dws_TAL', 'dws_EC', 'dws_pH', 'dws_Ca', 'dws_Mg',
                    'dws_Na', 'dws_Cl', 'dws_SO4', 'dws_days_diff', 'dws_dist_km', 'dws_P_modified',
                    'P_modified_same', 'dws_P_reliability'
                ] if c in dws.columns
            ]

            if 'Sample Date' in out.columns:
                out['Sample Date'] = _to_datetime_mixed(out['Sample Date'])

            dedup_dws = dws[key_cols + available_cols].drop_duplicates(subset=key_cols, keep='first')
            out = out.merge(dedup_dws, on=key_cols, how='left')
            return out

        def align_to_reference_schema(df: pd.DataFrame, ref_path: Path) -> pd.DataFrame:
            ref_cols = pd.read_csv(ref_path, nrows=0).columns.tolist()
            out = df.copy()
            for col in ref_cols:
                if col not in out.columns:
                    out[col] = np.nan
            return out[ref_cols]

        train_merged = merge_glorich_landsat(glorich_df, landsat_train, split_name='train')
        test_merged = merge_glorich_landsat(glorich_df, landsat_val, split_name='test')

        train_merged = merge_dws_features(train_merged, dws_train_df)
        test_merged = merge_dws_features(test_merged, dws_test_df)

        train_clean = clean_reliability_matrix(train_merged)
        test_clean = clean_reliability_matrix(test_merged)

        train_clean = align_to_reference_schema(train_clean, TRAIN_PATH)
        test_clean = align_to_reference_schema(test_clean, TEST_PATH)

        return train_clean, test_clean

    # Execute rebuild if flag is set
    print("\nRebuilding intermediate datasets...")
    final_glorich_rebuilt = rebuild_final_glorich_from_raw()
    print(f"  final_glorich: {final_glorich_rebuilt.shape}")

    landsat_train_rebuilt, landsat_val_rebuilt = rebuild_landsat_from_raw()
    print(f"  landsat_training: {landsat_train_rebuilt.shape}")
    print(f"  landsat_validation: {landsat_val_rebuilt.shape}")

    dws_matched_rebuilt, dws_test_rebuilt = rebuild_dws_from_raw()
    print(f"  dws_matched_all_columns: {dws_matched_rebuilt.shape}")
    print(f"  dws_test: {dws_test_rebuilt.shape}")

    print("\nRebuilding reliability matrices...")
    train_rebuilt, test_rebuilt = rebuild_train_test_reliability()
    print(f"  train_ALL+reliability: {train_rebuilt.shape}")
    print(f"  test_ALL+reliability: {test_rebuilt.shape}")

    print("\nRebuild complete. Loaded datasets are now in train_rebuilt / test_rebuilt.")
    print("To use rebuilt datasets, set train_df = train_rebuilt, test_df = test_rebuilt in the next cell.")

else:
    print("REBUILD_MODE=False: Skipping rebuild; will load pre-existing cleaned CSVs in next section.")

REBUILD_MODE=True: Regenerating datasets from raw sources...

Rebuilding intermediate datasets...
  final_glorich: (105393, 16)
  landsat_training: (9319, 10)
  landsat_validation: (200, 10)
  dws_matched_all_columns: (9319, 70)
  dws_test: (200, 70)

Rebuilding reliability matrices...
  train_ALL+reliability: (9171, 78)
  test_ALL+reliability: (200, 77)

Rebuild complete. Loaded datasets are now in train_rebuilt / test_rebuilt.
To use rebuilt datasets, set train_df = train_rebuilt, test_df = test_rebuilt in the next cell.


## 1) Data Preparation
Load and clean train/test reliability matrices through notebook-derived rules, and ingest upstream support datasets with the same lineage-aware cleaning functions.

In [19]:
def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['Sample Date'], errors='coerce')
    out['month'] = dt.dt.month
    out['year'] = dt.dt.year
    out['day_of_year'] = dt.dt.dayofyear
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12)
    out['doy_sin'] = np.sin(2 * np.pi * out['day_of_year'] / 365)
    out['doy_cos'] = np.cos(2 * np.pi * out['day_of_year'] / 365)
    out['wet_season'] = out['month'].isin([10, 11, 12, 1, 2, 3]).astype('Int64')
    return out


# Required reference artifacts for downstream reproducibility checks.
for p in [MAIN_SUBMISSION_PATH, ENSEMBLE_SOURCE_PATH]:
    require_file(p)

# Determine source: rebuilt (if REBUILD_MODE=True) vs. loaded/cleaned from CSV.
if REBUILD_MODE and 'train_rebuilt' in locals() and 'test_rebuilt' in locals():
    print("Using rebuilt datasets from raw sources...")
    train_df = train_rebuilt
    test_df = test_rebuilt
    supporting_datasets = {
        'final_glorich': final_glorich_rebuilt,
        'landsat_training': landsat_train_rebuilt,
        'landsat_validation': landsat_val_rebuilt,
        'dws_matched_all_columns': dws_matched_rebuilt,
        'dws_test': dws_test_rebuilt,
    }
else:
    print("Loading + cleaning pre-existing datasets...")
    train_df = load_and_clean_dataset('train_reliability', required=True)
    test_df = load_and_clean_dataset('test_reliability', required=True)

    supporting_datasets = {
        'final_glorich': load_and_clean_dataset('final_glorich', required=False),
        'landsat_training': load_and_clean_dataset('landsat_training', required=False),
        'landsat_validation': load_and_clean_dataset('landsat_validation', required=False),
        'dws_matched_all_columns': load_and_clean_dataset('dws_matched_all_columns', required=False),
        'dws_test': load_and_clean_dataset('dws_test', required=False),
    }

train_df = add_time_features(train_df)
test_df = add_time_features(test_df)

available_support = {
    key: (value.shape if isinstance(value, pd.DataFrame) else None)
    for key, value in supporting_datasets.items()
}

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
print('supporting datasets loaded:', available_support)
print('dws columns in train:', [c for c in train_df.columns if c.startswith('dws_') or c == 'P_modified_same'])
print('dws columns in test :', [c for c in test_df.columns if c.startswith('dws_') or c == 'P_modified_same'])

if 'dws_P_modified' in train_df.columns and 'dws_P_modified' in test_df.columns:
    print('dws_P_modified min/max (train):', float(np.nanmin(train_df['dws_P_modified'])), float(np.nanmax(train_df['dws_P_modified'])))
    print('dws_P_modified min/max (test):', float(np.nanmin(test_df['dws_P_modified'])), float(np.nanmax(test_df['dws_P_modified'])))
else:
    print('dws_P_modified not present yet; check rebuild merge_dws_features step.')

Using rebuilt datasets from raw sources...
train shape: (9171, 78)
test shape : (200, 77)
supporting datasets loaded: {'final_glorich': (105393, 16), 'landsat_training': (9319, 10), 'landsat_validation': (200, 10), 'dws_matched_all_columns': (9319, 70), 'dws_test': (200, 70)}
dws columns in train: ['dws_1st', 'dws_1st_dist', 'dws_TAL', 'dws_EC', 'dws_pH', 'dws_Ca', 'dws_Mg', 'dws_Na', 'dws_Cl', 'dws_SO4', 'dws_days_diff', 'dws_dist_km', 'dws_P_modified', 'P_modified_same', 'dws_P_reliability']
dws columns in test : ['dws_1st', 'dws_1st_dist', 'dws_TAL', 'dws_EC', 'dws_pH', 'dws_Ca', 'dws_Mg', 'dws_Na', 'dws_Cl', 'dws_SO4', 'dws_days_diff', 'dws_dist_km', 'dws_P_modified', 'dws_P_reliability']
dws_P_modified min/max (train): 0.2 0.2
dws_P_modified min/max (test): nan nan


## 2) Feature Engineering
Create shared and target-specific engineered features once for both train and test.

In [ ]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in ['dws_dist_km', 'dws_days_diff', 'dws_EC', 'dws_TAL', 'dws_P_modified', 'pet', 'SO4', 'Cl', 'SpecCond25C']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')

    if {'dws_dist_km', 'dws_days_diff'}.issubset(out.columns):
        decay = 1.0 + out['dws_dist_km'].abs() * 0.1 + out['dws_days_diff'].abs() * 0.01
        out['drp_dws_decay'] = out['dws_P_modified'] / decay
        if 'dws_EC' in out.columns:
            out['ec_dws_decay'] = out['dws_EC'] / decay
        if 'dws_TAL' in out.columns:
            out['tal_dws_decay'] = out['dws_TAL'] / decay

    if {'SpecCond25C', 'SO4', 'Cl'}.issubset(out.columns):
        out['ion_load_proxy'] = out['SO4'].fillna(0) + out['Cl'].fillna(0)
        out['ec_to_ion_ratio'] = out['SpecCond25C'] / (out['ion_load_proxy'] + 1e-6)

    if {'pet', 'wet_season'}.issubset(out.columns):
        out['pet_wet_interaction'] = out['pet'] * out['wet_season'].fillna(0)

    return out

train_fe = add_engineered_features(train_df)
test_fe = add_engineered_features(test_df)

print('engineered train shape:', train_fe.shape)
print('engineered test shape :', test_fe.shape)

## 3) Separate Model Training Blocks (3 Targets)
This keeps each target training independent. In `frozen_ensemble` mode, this section is skipped and the known final ensemble is used for exact reproducibility.

In [ ]:
def numeric_features(df: pd.DataFrame, drop_cols: set[str]) -> list[str]:
    cols = []
    for c in df.columns:
        if c in drop_cols:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

drop_common = {
    'Latitude', 'Longitude', 'Sample Date', 'date', 'geometry', 'STAT_ID',
    '_merge_landsat', '_merge_terra', 'Impute_Method',
}
drop_targets = set(TARGETS)

target_feature_blocks = {
    'Total Alkalinity': {'extra_drop': {'Electrical Conductance', 'Dissolved Reactive Phosphorus'}},
    'Electrical Conductance': {'extra_drop': {'Total Alkalinity', 'Dissolved Reactive Phosphorus'}},
    'Dissolved Reactive Phosphorus': {'extra_drop': {'Total Alkalinity', 'Electrical Conductance'}},
}

predictions = {}
training_report = {}

if REPRO_MODE == 'train_models':
    from sklearn.metrics import r2_score
    from sklearn.model_selection import train_test_split

    try:
        import lightgbm as lgb
    except Exception as exc:
        raise RuntimeError('train_models mode requires lightgbm installed.') from exc

    for target in TARGETS:
        local_drop = drop_common | drop_targets | target_feature_blocks[target]['extra_drop']
        feat_cols = numeric_features(train_fe, local_drop)

        X = train_fe[feat_cols].copy()
        y = pd.to_numeric(train_fe[target], errors='coerce')

        valid = y.notna()
        X = X.loc[valid].fillna(X.median(numeric_only=True))
        y = y.loc[valid]

        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE
        )

        if target == 'Dissolved Reactive Phosphorus':
            y_train_fit = np.log1p(np.clip(y_train, 0, None))
            y_val_fit = np.log1p(np.clip(y_val, 0, None))
        else:
            y_train_fit = y_train
            y_val_fit = y_val

        model = lgb.LGBMRegressor(
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
        )

        model.fit(
            X_train,
            y_train_fit,
            eval_set=[(X_val, y_val_fit)],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        val_pred = model.predict(X_val)
        if target == 'Dissolved Reactive Phosphorus':
            val_pred = np.expm1(val_pred)

        training_report[target] = {
            'n_features': len(feat_cols),
            'val_r2': float(r2_score(y_val, np.clip(val_pred, 0, None))),
        }

        X_test = test_fe[feat_cols].copy().fillna(X.median(numeric_only=True))
        pred = model.predict(X_test)
        if target == 'Dissolved Reactive Phosphorus':
            pred = np.expm1(pred)

        predictions[target] = np.clip(pred, 0, None)

else:
    frozen = pd.read_csv(ENSEMBLE_SOURCE_PATH)
    for target in TARGETS:
        predictions[target] = pd.to_numeric(frozen[target], errors='coerce').to_numpy()

    training_report = {
        'mode': 'frozen_ensemble',
        'source_file': str(ENSEMBLE_SOURCE_PATH),
        'note': 'Uses known final ensemble from EY_LightGBM_parallel lineage for exact reproducibility.'
    }

print(json.dumps(training_report, indent=2))

## 4) Submission Assembly
Assemble final submission and apply domain-safe clipping.

In [ ]:
submission = pd.DataFrame({
    'Latitude': test_fe['Latitude'],
    'Longitude': test_fe['Longitude'],
    'Sample Date': test_fe['Sample Date'],
})

for target in TARGETS:
    submission[target] = predictions[target]

# Domain clipping used in your prior pipeline family.
submission['Total Alkalinity'] = submission['Total Alkalinity'].clip(0, 400)
submission['Electrical Conductance'] = submission['Electrical Conductance'].clip(0, 1600)
submission['Dissolved Reactive Phosphorus'] = submission['Dissolved Reactive Phosphorus'].clip(0, 200)

submission.to_csv(OUTPUT_SUBMISSION_PATH, index=False)
print('Saved:', OUTPUT_SUBMISSION_PATH)
submission.head()

## 5) Provenance Checks
Check whether final targets were copied from candidate files (especially Electrical Conductance).

In [ ]:
def target_diff_counts(left: pd.DataFrame, right: pd.DataFrame, targets: list[str]) -> dict:
    out = {}
    for t in targets:
        l = pd.to_numeric(left[t], errors='coerce')
        r = pd.to_numeric(right[t], errors='coerce')
        out[t] = int((~np.isclose(l, r, equal_nan=True)).sum())
    return out

main_df = pd.read_csv(MAIN_SUBMISSION_PATH)
cand = {
    'submission_ensemble_cat_lgb.csv': pd.read_csv(ENSEMBLE_SOURCE_PATH),
}
if BEST_SUBMISSION_PATH.exists():
    cand['best_submission.csv'] = pd.read_csv(BEST_SUBMISSION_PATH)
if DWS_SUBMISSION_PATH.exists():
    cand['submission_dws_feat_eng.csv'] = pd.read_csv(DWS_SUBMISSION_PATH)

provenance = {name: target_diff_counts(main_df, df, TARGETS) for name, df in cand.items()}
print(json.dumps(provenance, indent=2))

## 6) Exact Reproducibility Check vs MAIN SUBMISSION
Compare row count, columns, ordering, and values; then save an audit report.

In [ ]:
generated_df = pd.read_csv(OUTPUT_SUBMISSION_PATH)
main_df = pd.read_csv(MAIN_SUBMISSION_PATH)

same_rows = generated_df.shape[0] == main_df.shape[0]
same_cols = list(generated_df.columns) == list(main_df.columns)

column_diff_counts = {}
if same_cols and same_rows:
    for col in generated_df.columns:
        l = generated_df[col]
        r = main_df[col]
        if pd.api.types.is_numeric_dtype(l) and pd.api.types.is_numeric_dtype(r):
            n_diff = int((~np.isclose(l, r, equal_nan=True)).sum())
        else:
            n_diff = int((l.fillna('__NA__') != r.fillna('__NA__')).sum())
        column_diff_counts[col] = n_diff
    exact_match = all(v == 0 for v in column_diff_counts.values())
else:
    exact_match = False

audit = {
    'generated_file': str(OUTPUT_SUBMISSION_PATH),
    'main_reference_file': str(MAIN_SUBMISSION_PATH),
    'same_row_count': bool(same_rows),
    'same_column_order': bool(same_cols),
    'exact_match': bool(exact_match),
    'column_diff_counts': column_diff_counts,
    'likely_cause_if_not_exact': (
        'Different model weights/features or DWS overrides, or train_models mode was used.'
        if not exact_match else 'None (exact match).'
    ),
}

with open(AUDIT_PATH, 'w', encoding='utf-8') as f:
    json.dump(audit, f, indent=2)

print(json.dumps(audit, indent=2))
print('Saved audit:', AUDIT_PATH)